# AIOps — Laboratório introdutório PCA, K-Means e t-SNE (Wine)

Material complementar ao primeiro post. Dataset Wine do scikit-learn. Execute no VS Code com kernel Python (AIOps).

Os gráficos e indicadores dependem da execução local; os números divulgados no post correspondem à execução de 02/10/2026.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from pathlib import Path

SEED = 42
print("Ambiente preparado")

Ambiente preparado


## 1. Reprodução corrigida — Wine
13 atributos químicos. Separar antes de aprender escalas ou PCA evita vazamento. O K-Means deve receber **PCA transformado**; aplicar `predict` a conjuntos de validação/teste no **mesmo espaço**. Os rótulos do Wine são usados só para referência externa, jamais durante treinamento.

In [2]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
scaler = StandardScaler().fit(X_train)
X_train_std = scaler.transform(X_train)
X_test_std = scaler.transform(X_test)
pca = PCA(n_components=2).fit(X_train_std)
X_train_pca = pca.transform(X_train_std)
X_test_pca = pca.transform(X_test_std)
km_wine = KMeans(n_clusters=3, random_state=SEED, n_init=10).fit(X_train_pca)
labels_train = km_wine.labels_
labels_test = km_wine.predict(X_test_pca)
wine_metrics = {"variancia_pca_2": float(pca.explained_variance_ratio_.sum()), "silhouette_treino": float(silhouette_score(X_train_pca, labels_train)), "silhouette_teste": float(silhouette_score(X_test_pca, labels_test)), "ARI_teste_somente_referencia": float(adjusted_rand_score(y_test, labels_test))}
print(pd.Series(wine_metrics).round(4))
assert X_train_pca.shape[1] == 2 and X_test_pca.shape[1] == 2

variancia_pca_2                 0.5496
silhouette_treino               0.5779
silhouette_teste                0.5252
ARI_teste_somente_referencia    0.8575
dtype: float64


In [3]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(X_train_pca[:,0], X_train_pca[:,1], c=labels_train, cmap="viridis", alpha=.8)
ax.scatter(km_wine.cluster_centers_[:,0], km_wine.cluster_centers_[:,1], marker="X", s=160, c="black", label="Centroides")
ax.set(title="Wine: PCA + K-Means (treinamento)", xlabel="Componente 1", ylabel="Componente 2")
ax.legend()
plt.tight_layout()
plt.show()

/tmp/ipykernel_433/802675934.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. t-SNE para visualização (não para pontuar novas VMs)
Um mapa t-SNE não tem eixos físicos interpretáveis, e distâncias grandes entre ilhas nem sempre representam distâncias reais. A aplicação padrão do scikit-learn **não dispõe de `transform`**. A cor é o grupo definido no PCA; t-SNE aqui não participa do treinamento do K-Means.

In [4]:
X_wine_all_std = scaler.transform(X)
embedding = TSNE(n_components=2, perplexity=20, random_state=SEED, init="pca", learning_rate="auto", max_iter=700).fit_transform(X_wine_all_std)
fig, ax=plt.subplots(figsize=(7,5))
ax.scatter(embedding[:,0], embedding[:,1], c=y, cmap="viridis", alpha=.8)
ax.set(title="Wine: t-SNE exploratório (cores = classes conhecidas)", xlabel="t-SNE 1 (arbitrário)", ylabel="t-SNE 2 (arbitrário)")
plt.tight_layout()
plt.show()

/tmp/ipykernel_433/2326469271.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
